In [38]:
# @title
from IPython.display import display, HTML

display(HTML("""
<div style="
  margin: 18px 0 26px 0;
  padding: 20px 24px;
  border-radius: 16px;
  background: linear-gradient(135deg, #f0f4f8 0%, #d9e2ec 100%);
  border-bottom: 4px solid #334e68;
  box-shadow: 0 4px 16px rgba(51,78,104,0.15);
  color: #334e68;
">

  <div style="text-align:center;">

    <div style="
      font-size: 26px;
      font-weight: 700;
      letter-spacing: -0.3px;
      line-height: 1.2;
      margin: 0;
      color: #334e68;
    ">
      🏠 Projet de Machine Learning
    </div>

    <div style="
      margin-top: 6px;
      font-size: 18px;
      color: #486581;
      line-height: 1.4;
      font-weight: 500;
    ">
      Prédiction des prix des maisons
      <span style="
        display:inline-block;
        margin-left: 8px;
        padding: 2px 10px;
        border-radius: 999px;
        background: #334e68;
        color: #ffffff;
        font-weight: 600;
        font-size:13px;
        vertical-align: middle;
      ">
        Régression
      </span>
    </div>

    <div style="
      margin-top: 8px;
      font-size: 14px;
      color: #627d98;
      font-weight: 400;
    ">
      EDA · Gestion des NaN · Encodage · Création de features · Standardisation
    </div>

    <div style="
      margin-top: 10px;
      height: 3px;
      width: 80px;
      background: #334e68;
      border-radius: 999px;
      margin-left: auto;
      margin-right: auto;
    "></div>

  </div>
</div>
"""))

# Analyse Exploratoire des Données  et feature engineering (EDA)



## Introduction 

L’objectif de ce notebook est d’explorer le jeu de données House Prices afin de mieux comprendre sa structure, d’identifier les variables importantes, les valeurs manquantes et les éventuelles anomalies.
Cette analyse exploratoire constitue une étape essentielle avant la phase de feature engineering et de modélisation. Les observations réalisées dans ce notebook guideront les décisions prises pour la préparation des données et la construction du modèle de prédiction.

## Importer les librairies et chargement des données

In [39]:
# ============================================
# IMPORT DES LIBRAIRIES
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# Afficher toutes les colonnes
pd.pandas.set_option('display.max_columns', None)

# Limiter les flottants à 3 décimales
pd.set_option('display.float_format', lambda x: '{:.3f}'.format(x))


sns.set_style("whitegrid")


custom_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
                  '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
sns.set_palette(custom_palette)

# ============================================
# LIBRAIRIES SUPPLÉMENTAIRES
# ============================================

from scipy import stats
from scipy.stats import norm, skew

In [40]:
    
# On utilise le dossier "data" à la racine du projet
data = pd.read_csv('../data/train.csv')



## Aperçu des données 

In [41]:
#check the numbers of samples and features
print(f"The train data size before dropping Id feature is : {data.shape}")



#Save the 'Id' column
data_ID = data['Id']


#Now drop the  'Id' colum since it's unnecessary for  the prediction process.
data.drop("Id", axis = 1, inplace = True)


#check again the data size after dropping the 'Id' variable
print(f"The  data size after dropping Id feature is  : {data.shape}")



The train data size before dropping Id feature is : (1460, 81)
The  data size after dropping Id feature is  : (1460, 80)


In [ ]:
data.head()

In [ ]:
data.describe(include="all")

In [ ]:
data.info()

## Valeurs manquantes

### Analyse des valeurs manquantes

In [ ]:
# ============================================
# ANALYSE DES VALEURS MANQUANTES (AVEC TYPE)
# ============================================

# 1. Identifier les colonnes avec des valeurs manquantes
features_with_na = [feature for feature in data.columns if data[feature].isnull().sum() > 0]

# 2. Créer un DataFrame récapitulatif
missing = data[features_with_na].isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct,
    'Type': data[features_with_na].dtypes
}).sort_values('Pourcentage (%)', ascending=False)

print("="*60)
print("VALEURS MANQUANTES PAR COLONNE")
print("="*60)
print(f"Nombre total de colonnes avec des NaN : {len(missing_df)}")
print("\n")
print(missing_df)

**Étant donné le nombre élevé de valeurs manquantes, il convient d’étudier leur lien avec le prix de vente.**


### Impact des valeurs manquantes sur le prix 

In [ ]:
# ============================================
# IMPACT DES VALEURS MANQUANTES SUR LE PRIX
# ============================================

# 2. Pour chaque colonne, visualiser l'impact sur le prix
for feature in features_with_na:
    dataset_copy = data.copy()
    dataset_copy[feature] = np.where(dataset_copy[feature].isnull(), 1, 0)
    
    # Graphique avec couleurs
    plt.figure(figsize=(8, 5))
    dataset_copy.groupby(feature)['SalePrice'].median().plot.bar(
        color=['skyblue', 'salmon']  # ← Maintenant bien indenté
    )
    plt.title(f'Impact des valeurs manquantes de {feature} sur le prix')
    plt.xlabel('0 = présent, 1 = manquant')
    plt.ylabel('Prix médian (en dollars)')
    plt.xticks([0, 1], ['Présent', 'Manquant'], rotation=0)
    plt.show()

L’analyse montre que, pour la plupart des variables, l’absence d’un équipement (piscine, cheminée, garage, sous‑sol) est liée à un prix plus bas, ce qui est attendu. Cependant, pour Alley, Fence, LotFrontage, MasVnrArea et MiscFeature, les maisons avec des valeurs manquantes ont un prix médian plus élevé. Ce résultat contre‑intuitif s’explique par le fait que ces manquantes sont souvent des indicateurs indirects d’autres caractéristiques valorisantes : quartier plus moderne (Alley), terrain atypique ou de grande taille (LotFrontage), finitions complexes ou haut de gamme (MasVnrArea), ou absence d’équipements secondaires typique des maisons modestes (MiscFeature, Fence). Ces valeurs manquantes ne sont donc pas aléatoires et doivent être conservées comme une catégorie explicite (« Non renseigné ») pour ne pas perdre l’information qu’elles portent. Le cas d’Electrical, avec une seule valeur manquante, est un cas isolé sans signification statistique.

## Analyse des variables numériques 

In [ ]:
# ============================================
# ANALYSE DES VARIABLES NUMÉRIQUES
# ============================================

# 1. Identifier les variables numériques (version robuste)
numerical_features = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Nombre de variables numériques : {len(numerical_features)}')
print(f'Colonnes : {numerical_features[:10]}')  # Les 10 premières

# 2. Visualiser les premières lignes des variables numériques
data[numerical_features].head()

## Corrélations

### Analyse des corrélations

In [ ]:

# ============================================
# ANALYSE DES CORRÉLATIONS
# ============================================

# 2. MATRICE DE CORRÉLATION
# -------------------------
numVar = data.select_dtypes(include=[np.number])
cor_numVar = numVar.corr(method='pearson')

# 3. TRIER PAR CORRÉLATION AVEC SALEPRICE
# ---------------------------------------
cor_sorted = cor_numVar['SalePrice'].sort_values(ascending=False)

print("="*60)
print("TOP 15 CORRÉLATIONS AVEC SALEPRICE")
print("="*60)
for feature, corr in cor_sorted.head(15).items():
    if feature != 'SalePrice':
        print(f"{feature:20} : {corr:.4f}")

# 4. GARDER UNIQUEMENT LES CORRÉLATIONS ÉLEVÉES (> 0.5)
# ----------------------------------------------------
CorHigh = cor_sorted[abs(cor_sorted) > 0.5].index.tolist()
print(f"\nVariables avec |corrélation| > 0.5 : {len(CorHigh)}")

# 5. HEATMAP DES CORRÉLATIONS
# ----------------------------
cor_numVar_high = cor_numVar.loc[CorHigh, CorHigh]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(cor_numVar_high, dtype=bool), k=1)

sns.heatmap(
    cor_numVar_high,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8}
)

ax.set_title("Corrélations avec SalePrice (|r| > 0.5)", fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



La matrice de corrélation met en évidence les variables les plus liées au prix de vente : OverallQual (0,79), GrLivArea (0,71), GarageCars (0,64), GarageArea (0,62), TotalBsmtSf et 1stFlrSF (0,61) sont les prédicteurs les plus forts. Des variables comme FullBath et TotRmsAbvGrd présentent également des corrélations significatives (> 0,5). Cependant, plusieurs de ces variables sont fortement corrélées entre elles, notamment TotalBsmtSF avec 1stFlrSF (0,82) et GarageArea avec GarageCars (0,88), ce qui indique un risque de multicolinéarité. Ces redondances devront être traitées lors de la modélisation, par exemple en sélectionnant une variable parmi chaque groupe ou en créant des caractéristiques agrégées (surface totale, etc.). Les variables temporelles (YearBuilt, YearRemodAdd) montrent aussi une influence notable et pourraient être transformées en âge pour mieux capturer leur effet.

### Analyse des variables temporelles 

In [ ]:
# ============================================
# ANALYSE DES VARIABLES TEMPORELLES
# ============================================

# 1. Identifier les colonnes avec "Year" ou "Yr"
year_feature = [feature for feature in numerical_features if 'Yr' in feature or 'Year' in feature]

print("Variables temporelles identifiées :")
print(year_feature)
# let's explore the content of these year variables
for feature in year_feature:
    print(feature, data[feature].unique())

# 2. Visualiser ces variables
data[year_feature].head()

In [ ]:
## Lets analyze the Temporal Datetime Variables
## We will check whether there is a relation between year the house is sold and the sales price

data.groupby('YrSold')['SalePrice'].median().plot()
plt.xlabel('Year Sold')
plt.ylabel('Median House Price')
plt.title("House Price vs YearSold")

L'allure de ce graphique est contre-intuitive, mais elle s'explique parfaitement. Entre 2006 et 2007, la survalorisation des biens immobiliers aux États-Unis, alimentée par une demande bien supérieure à l'offre, a généré un pic artificiel en 2007. Ce sommet a ensuite cédé la place à une chute brutale jusqu'en 2010, due à l'effondrement du marché provoqué par la crise des subprimes.

### Analyse de l'impact de l'age des maisons sur le prix 

In [ ]:
# ============================================
# ANALYSE DE L'IMPACT DE L'ÂGE SUR LE PRIX
# ============================================

year_feature = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'YrSold']

for feature in year_feature:
    if feature!='YrSold':
        dataset=data.copy()
        ## We will capture the difference between year variable and year the house was sold for
        dataset[feature]=dataset['YrSold']-dataset[feature]

        plt.scatter(dataset[feature],dataset['SalePrice'])
        plt.xlabel(feature)
        plt.ylabel('SalePrice')
        plt.show()

L'analyse révèle que l'âge de la maison exerce un impact globalement négatif et non linéaire sur le prix, un effet considérablement modéré par l'historique des rénovations et profondément brouillé par le contexte macroéconomique de 2006 à 2010, où le pic de la bulle immobilière en 2007 et l'effondrement qui a suivi ont éclipsé la tendance naturelle au vieillissement.

### Relation prix et variable discrète

In [ ]:
## Numerical variables are usually of 2 type
## 1. Continous variable and Discrete Variables

discrete_feature=[feature for feature in numerical_features if len(data[feature].unique())<25 and feature not in year_feature]
print("Discrete Variables Count: {}".format(len(discrete_feature)))

In [ ]:
discrete_feature

In [ ]:
data[discrete_feature].head()

In [ ]:

for feature in discrete_feature:
    dataset=data.copy()
    dataset.groupby(feature)['SalePrice'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('SalePrice')
    plt.title(feature)
    plt.show()

Les variables discrètes, comme la qualité globale, le nombre de garages ou de chambres, agissent par paliers : chaque palier supplémentaire fait monter le prix, mais parfois plus fortement que d’autres. La qualité est de loin la plus influente. Pour le feature engineering, il faut les traiter comme des catégories ordonnées (et non comme des nombres continus) 

### Distribution des variables continues 

In [ ]:
continuous_feature=[feature for feature in numerical_features if feature not in discrete_feature+year_feature]
print("Continuous feature Count {}".format(len(continuous_feature)))

In [ ]:
## Lets analyse the continuous values by creating histograms to understand the distribution

for feature in continuous_feature:
    dataset=data.copy()
    data[feature].hist(bins=25)
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.title(feature)
    plt.show()


In [ ]:
for feature in continuous_feature:
    dataset=data.copy()
    if 0 in dataset[feature].unique():
        pass
    else:
        dataset[feature]=np.log(dataset[feature])
        data.boxplot(column=feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()
        

**Analyse des variables continues**

Après avoir regardé les statistiques descriptives des variables continues, on observe que beaucoup présentent **une forte asymétrie** et **des valeurs extrêmes**.

**Exemples marquants :**

• **LotArea** : 
  • Moyenne = 10 516
  • Maximum = 215 245
  • → Quelques terrains gigantesques tirent la moyenne vers le haut

• **GrLivArea** : 
  • Min = 334
  • Max = 5 642
  • → Queue de distribution très longue

• **TotalBsmtSF** : 
  • Min = 0
  • Max = 6 110
  • → Forte concentration vers 0 avec quelques valeurs très élevées

**Ce que cela signifie :**

• Les variables **ne sont pas symétriques**
• Les **modèles linéaires** (qui supposent une distribution normale) **risquent d'être perturbés** par ces valeurs extrêmes
• L'**écart-type** est souvent très grand → forte variabilité
• Les **quantiles** montrent que la plupart des maisons sont dans une fourchette raisonnable, mais **quelques-unes sortent complètement du lot**

**Etudions la relation entre le prix des maisons et la surface habitable puisque c'est l'une des variables les plus corrélées avec salePrice et verifions s'il y'a des outlyers a supprimé avant la modélisation**

In [ ]:
# Création du graphique
fig, ax = plt.subplots(figsize=(12, 8))

# Nuage de points
ax.scatter(data['GrLivArea'], data['SalePrice'], 
           alpha=0.6, 
           color='steelblue', 
           edgecolors='white', 
           linewidth=0.5,
           s=50)

# Ligne de tendance (régression linéaire)
z = np.polyfit(data['GrLivArea'], data['SalePrice'], 1)
p = np.poly1d(z)
ax.plot(data['GrLivArea'].sort_values(), 
        p(data['GrLivArea'].sort_values()), 
        color='red', 
        linewidth=2, 
        label=f'Tendance (pente = {z[0]:.2f})')

# Personnalisation
ax.set_xlabel('Surface Habitable (GrLivArea) - pieds carrés', fontsize=14)
ax.set_ylabel('Prix de Vente (SalePrice) - $', fontsize=14)
ax.set_title('Relation entre Surface Habitable et Prix de Vente', fontsize=16, fontweight='bold')

# Ajout de la corrélation
correlation = data['GrLivArea'].corr(data['SalePrice'])
ax.text(0.05, 0.95, f'Corrélation = {correlation:.3f}', 
        transform=ax.transAxes, 
        fontsize=14, 
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Identification des valeurs aberrantes de GrLivArea**

In [ ]:
# ============================================
# IDENTIFICATION DES VALEURS ABERRANTES
# ============================================

# Condition : Surface > 4000 ET Prix < 200000
outliers_condition = (data['GrLivArea'] > 4000) & (data['SalePrice'] < 200000)

# Identifier les indices
outliers_idx = data[outliers_condition].index.tolist()

# Afficher les valeurs aberrantes
print("=== VALEURS ABERRANTES IDENTIFIÉES ===")
print(f"Nombre de valeurs aberrantes : {len(outliers_idx)}")
print("\nDétail des valeurs aberrantes :")
print(data[outliers_condition][['GrLivArea', 'SalePrice']])

# Si vous voulez voir toutes les colonnes
# print(data.loc[outliers_idx].head())

In [ ]:
# Supression des outliers
data = data.drop(index=outliers_idx)

In [ ]:
data.shape[0]

**Relation entre Surface Habitable et Prix de Vente**

**Objectif :** Visualiser et analyser la relation entre la surface habitable (GrLivArea) et le prix de vente (SalePrice) pour comprendre l'impact de cette variable sur la prédiction.

---

**Graphique : Nuage de points avec ligne de tendance**

• **Chaque point représente une maison**
  • Axe X : Surface habitable (GrLivArea) en pieds carrés
  • Axe Y : Prix de vente (SalePrice) en dollars

• **Ligne de tendance (rouge)**
  • Pente = 107.13
  • Interprétation : En moyenne, le prix augmente de 107 $ par pied carré supplémentaire
  • La relation est globalement linéaire

• **Corrélation affichée : 0.709**
  • Relation positive forte
  • Plus la surface est grande, plus le prix est élevé
  • Variable importante pour la prédiction

---

**Observations clés**

**Points positifs :**
• La majorité des points suit la tendance
• La relation est claire et significative
• Les points sont bien répartis sur l'ensemble des surfaces

**Points problématiques :**
• Présence de **deux valeurs aberrantes** :
  • Indice 523 : GrLivArea = 4 676 ft² pour un prix de 184 750 $
  • Indice 1298 : GrLivArea = 5 642 ft² pour un prix de 160 000 $
• Ces deux maisons ont une très grande surface mais un prix relativement bas
• Elles ne suivent pas la tendance générale

---

**Traitement des valeurs aberrantes**

**Pourquoi les supprimer ?**

• Ces points ne suivent pas la tendance générale
• Ils peuvent fausser les coefficients des modèles linéaires
• Ils pourraient tirer la ligne de régression vers le bas
• Cela affecterait la précision des prédictions sur les surfaces normales

**Suppression :**
• Nous avons supprimé ces 2 lignes du dataset
• Avant : 1 460 observations
• Après : 1 458 observations
• Perte d'information : 0.14% (négligeable)

---

**Interprétation finale**

• La surface habitable est un **facteur déterminant** du prix de vente
• Corrélation de 0.709 → variable très importante pour la prédiction
• La relation est positive : plus de surface = prix plus élevé
• Les valeurs aberrantes ont été supprimées pour ne pas fausser l'analyse

**Conclusion :**
GrLivArea sera une variable clé dans notre modèle de prédiction.
Son impact est significatif, comme le montre la corrélation et la pente de la tendance.

### Analyse des variables catégorielles

In [ ]:
## Categorical Variables
categorical_features=[feature for feature in data.columns if data[feature].dtypes=='O']
categorical_features


In [ ]:
for feature in categorical_features:
    print('The feature is {} and number of categories are {}'.format(feature,len(data[feature].unique())))

In [ ]:
data[categorical_features].head()

In [ ]:
for feature in categorical_features:
    dataset=data.copy()
    dataset.groupby(feature)['SalePrice'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('SalePrice')
    plt.title(feature)
    plt.show()

**Analyse des variables catégorielles**

Les variables catégorielles comme le quartier, la qualité des matériaux ou le type de garage influencent fortement le prix de vente. Leur impact se traduit par des écarts de prix importants entre modalités et par des valeurs manquantes qui, dans ce jeu de données, sont souvent informatives (absence d'un équipement).

**Observations clés :**

• **Le quartier (Neighborhood)** : 
  • Écart de prix important entre les quartiers
  • StoneBr : prix moyen de 310 000 $
  • MeadowV : prix moyen de 88 000 $
  • → L'emplacement géographique est un facteur déterminant

• **La qualité extérieure (ExterQual)** : 
  • Ex (Excellente) : prix moyen de 250 000 $
  • Fa (Faible) : prix moyen de 90 000 $
  • → La qualité perçue de la maison est très influente

• **Le type de garage (GarageType)** : 
  • Attchd (Garage attaché) : prix moyen plus élevé
  • Detchd (Garage détaché) : prix moyen intermédiaire
  • None (Pas de garage) : prix moyen plus bas
  • → Les valeurs manquantes signifient "pas de garage"

**Ce que cela signifie :**

• Les variables catégorielles ont un pouvoir prédictif important
• Les valeurs manquantes sont souvent informatives (absence d'une caractéristique)
• Les écarts de prix entre modalités peuvent être significatifs

**Stratégie d'encodage retenue**

Pour encoder nos variables catégorielles, nous avons distingué deux cas :

**1. Variables ordinales (avec ordre logique)**

• **Variables concernées** : KitchenQual, ExterQual, ExterCond, BsmtQual, BsmtCond, HeatingQC, GarageQual, GarageCond, PoolQC, FireplaceQu
• **Méthode** : Mapping personnalisé

Ces variables ont un ordre naturel (Ex > Gd > TA > Fa > Po). Nous leur avons attribué des valeurs numériques qui respectent cette hiérarchie (Ex → 5, Gd → 4, TA → 3, Fa → 2, Po → 1, None → 0).

**Pourquoi ?** LabelEncoder aurait attribué un ordre alphabétique arbitraire, ce qui ne reflète pas la réalité. Le mapping préserve la logique métier.

**2. Variables nominales (sans ordre logique)**

• **Variables concernées** : Les autres variables catégorielles (quartier, zone, etc.)
• **Méthode** : LabelEncoder

Chaque catégorie est transformée en un nombre entier (ex: "RL" → 0, "RM" → 1, "FV" → 2).

**Pourquoi ?** Les modèles à arbres (CatBoost) ne sont pas sensibles à l'ordre des nombres. LabelEncoder est simple, efficace et évite la création de centaines de colonnes qu'aurait générées One-Hot Encoding.

**Gestion des valeurs manquantes :**

Nous avons conservé les NaN comme une catégorie explicite "None" (ex: GarageType = None signifie "pas de garage"). Cela permet de tirer parti de l'information contenue dans l'absence d'un équipement.

**Gestion des valeurs inconnues dans le test :**

Si une catégorie apparaît dans le test mais pas dans le train, elle est remplacée par la catégorie la plus fréquente, garantissant ainsi la cohérence entre les ensembles.

## Description de la variable cible 

In [ ]:
# Description de la variable cible
data['SalePrice'].describe()

In [ ]:
print(f"Skewness originale : {data['SalePrice'].skew():.2f}")

In [ ]:
# ============================================
# DISTRIBUTION DE SALEPRICE : ORIGINAL VS LOG
# ============================================

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))

# Avant transformation
sns.histplot(data["SalePrice"], color='b', kde=True, ax=axes[0])
axes[0].set_title(f'Distribution - SalePrice original\nSkewness = {data["SalePrice"].skew():.4f}')

# Après transformation log
sns.histplot(np.log1p(data['SalePrice']), color='coral', kde=True, ax=axes[1])
axes[1].set_title(f'SalePrice après log1p\nSkewness = {np.log1p(data["SalePrice"]).skew():.4f}')
axes[1].set_xlabel('log1p(SalePrice)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# QQ-PLOT : ORIGINAL VS LOG
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avant transformation
stats.probplot(data['SalePrice'], plot=axes[0])
axes[0].set_title('QQ-Plot - SalePrice original')
axes[0].get_lines()[0].set(color='steelblue', markersize=3)
axes[0].get_lines()[1].set(color='red')

# Après transformation
stats.probplot(np.log1p(data['SalePrice']), plot=axes[1])
axes[1].set_title('QQ-Plot - log1p(SalePrice)')
axes[1].get_lines()[0].set(color='coral', markersize=3)
axes[1].get_lines()[1].set(color='red')

plt.tight_layout()
plt.show()

**Analyse de la variable cible : SalePrice**

La variable cible est le prix de vente des maisons, exprimé en dollars. Sa distribution et ses caractéristiques sont essentielles à comprendre avant toute modélisation.

**Statistiques descriptives :**

• **Moyenne** : 180 921 $
• **Médiane** : 163 000 $
• **Écart-type** : 79 443 $
• **Minimum** : 34 900 $
• **Maximum** : 755 000 $
• **Skewness (asymétrie)** : 1.88

**Ce que ces statistiques nous disent :**

• Le prix moyen est de 180 921 $, mais la médiane est plus basse (163 000 $)
• Cela indique une asymétrie à droite : quelques maisons très chères tirent la moyenne vers le haut
• L'écart-type est élevé (79 443 $), ce qui montre une grande variabilité des prix
• La fourchette de prix est large : de 34 900 $ à 755 000 $

**Visualisation de la distribution :**

• **Histogramme original** : La distribution est asymétrique à droite (skewness = 1.88)
• **Histogramme après transformation log** : La distribution devient plus symétrique (skewness = 0.12)
• **QQ-Plot original** : Les points s'écartent de la ligne rouge, indiquant une non-normalité
• **QQ-Plot après log** : Les points sont mieux alignés, indiquant une meilleure normalité

**Interprétation :**

La transformation logarithmique (log) permet de :
→ Réduire l'asymétrie de la distribution
→ Rendre la distribution plus proche d'une loi normale
→ Améliorer la performance des modèles linéaires

**Décision pour la modélisation :**

Notre modèle final est CatBoost (modèle à arbres), qui est robuste aux distributions asymétriques.
Nous avons donc conservé SalePrice tel quel, sans transformation log.
Cette décision a été prise après avoir testé plusieurs modèles.

In [ ]:
# ============================================
# STATISTIQUES : ORIGINAL VS LOG
# ============================================

print("="*60)
print("COMPARAISON DES STATISTIQUES")
print("="*60)

print(f"Skewness originale  : {data['SalePrice'].skew():.4f}")
print(f"Kurtosis originale  : {data['SalePrice'].kurt():.4f}")
print("---" + "-"*40)
print(f"Skewness après log  : {np.log1p(data['SalePrice']).skew():.4f}")
print(f"Kurtosis après log  : {np.log1p(data['SalePrice']).kurt():.4f}")

## Feature engineering

### Imputation des valeurs manquantes par "None"

In [ ]:
cols_none = [
    "PoolQC", 
    "MiscFeature", 
    "Fence", 
    "FireplaceQu",
    "Alley",          # pas d'allée
    "GarageType",     # pas de garage
    "GarageFinish",   # pas de garage
    "GarageQual",     # pas de garage
    "GarageCond",     # pas de garage
    "BsmtQual",       # pas de sous-sol
    "BsmtCond",       # pas de sous-sol
    "BsmtExposure",   # pas de sous-sol
    "BsmtFinType1",   # pas de sous-sol
    "BsmtFinType2",   # pas de sous-sol
    "MasVnrType"      # pas de maçonnerie
]

data[cols_none] = data[cols_none].fillna("None")


**Gestion des valeurs manquantes**

Une première approche consiste à identifier les colonnes où les valeurs manquantes signifient en réalité l'absence de la caractéristique.

**Colonnes concernées :**

• **Caractéristiques extérieures :**
  • PoolQC : pas de piscine
  • Fence : pas de clôture
  • Alley : pas d'allée

• **Garage :**
  • GarageType, GarageFinish, GarageQual, GarageCond : pas de garage

• **Sous-sol :**
  • BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2 : pas de sous-sol

• **Autres :**
  • MiscFeature : pas de caractéristique supplémentaire
  • FireplaceQu : pas de cheminée
  • MasVnrType : pas de parement en maçonnerie

**Traitement appliqué :**

Nous avons remplacé les NaN par "None" pour ces 15 colonnes.

**Pourquoi cette approche ?**

• Les NaN signifient "absence de la caractéristique" et non une donnée manquante
• "None" est une catégorie explicite qui conserve l'information
• Le modèle pourra apprendre l'impact de l'absence de ces équipements sur le prix

**Exemple :**

• GarageType = None → la maison n'a pas de garage
• PoolQC = None → la maison n'a pas de piscine
• BsmtQual = None → la maison n'a pas de sous-sol

### Création de nouvelles variables 

In [ ]:
def create_all_features(df):
    """Crée les features les plus utiles (version simplifiée)"""
    df_feat = df.copy()
    
    
    # 1. QualityArea (découverte ML Mastery)
    df_feat['QualityArea'] = df_feat['OverallQual'] * df_feat['GrLivArea']
    
    # 2. TotalSF (surface totale habitable)
    df_feat['TotalSF'] = df_feat['TotalBsmtSF'] + df_feat['1stFlrSF'] + df_feat['2ndFlrSF']
    
    # 3. HouseAge (âge de la maison)
    df_feat['HouseAge'] = df_feat['YrSold'] - df_feat['YearBuilt']
    
    # 4. RemodAge (âge depuis la rénovation)
    df_feat['RemodAge'] = df_feat['YrSold'] - df_feat['YearRemodAdd']
    
    # 5. TotalBath (salles de bain totales)
    df_feat['TotalBath'] = (df_feat['FullBath'] + 0.5*df_feat['HalfBath'] +
                            df_feat['BsmtFullBath'] + 0.5*df_feat['BsmtHalfBath'])
    
    # 6. TotalPorchSF (surface des porches)
    df_feat['TotalPorchSF'] = (df_feat['OpenPorchSF'] + df_feat['EnclosedPorch'] + 
                                df_feat['3SsnPorch'] + df_feat['ScreenPorch'])
    
   
    return df_feat

print(" Fonction create_all_features() définie (version simplifiée)")

In [ ]:
data_feat = create_all_features(data)

**Création de nouvelles features**

Pour enrichir notre jeu de données, nous avons créé 6 nouvelles variables à partir des variables existantes.

**1. QualityArea**

• **Formule** : `OverallQual × GrLivArea`
• **Description** : Combine la qualité globale avec la surface habitable
• **Pourquoi** : Une grande maison de mauvaise qualité n'a pas la même valeur qu'une petite maison de bonne qualité
• **Source** : Découverte dans la documentation ML Mastery

**2. TotalSF**

• **Formule** : `TotalBsmtSF + 1stFlrSF + 2ndFlrSF`
• **Description** : Surface totale de la maison (sous-sol + rez-de-chaussée + étage)
• **Pourquoi** : La surface totale est plus pertinente que les surfaces séparées

**3. HouseAge**

• **Formule** : `YrSold - YearBuilt`
• **Description** : Âge de la maison au moment de la vente
• **Pourquoi** : Une maison ancienne peut se déprécier

**4. RemodAge**

• **Formule** : `YrSold - YearRemodAdd`
• **Description** : Âge depuis la dernière rénovation
• **Pourquoi** : Une maison récemment rénovée a plus de valeur

**5. TotalBath**

• **Formule** : `FullBath + 0.5×HalfBath + BsmtFullBath + 0.5×BsmtHalfBath`
• **Description** : Nombre total de salles de bain (pondéré)
• **Pourquoi** : Une salle de bain complète vaut plus qu'une demi-salle

**6. TotalPorchSF**

• **Formule** : `OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch`
• **Description** : Surface totale des porches
• **Pourquoi** : Les espaces extérieurs contribuent à la valeur de la maison

**Résumé :**

• 6 nouvelles features créées
• Ces variables combinent des informations pour capturer des phénomènes plus complexes
• Elles enrichissent le jeu de données pour la modélisation
| Feature | Formule | Description |
|---------|---------|-------------|
| QualityArea | OverallQual × GrLivArea | Qualité × Surface habitable |
| TotalSF | TotalBsmtSF + 1stFlrSF + 2ndFlrSF | Surface totale |
| HouseAge | YrSold - YearBuilt | Âge de la maison |
| RemodAge | YrSold - YearRemodAdd | Âge depuis la rénovation |
| TotalBath | FullBath + 0.5×HalfBath + BsmtFullBath + 0.5×BsmtHalfBath | Salles de bain totales |
| TotalPorchSF | OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch | Surface totale des porches |

### Séparation de la base en train et test 

In [ ]:
# ============================================
# SPLIT TRAIN/TEST 
# ============================================

from sklearn.model_selection import train_test_split

# 1. Séparer X (features) et y (cible)
X = data_feat.drop('SalePrice', axis=1)
y = data_feat['SalePrice']

# 2. Split 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,          # 20% pour le test
    random_state=42
)

print(f"Train: {X_train.shape[0]} lignes")
print(f"Test: {X_test.shape[0]} lignes")

**Séparation des données : Train / Test**

Nous avons séparé notre jeu de données en deux ensembles :

**Méthodologie :**

• **Train** : 80% des données (1 168 maisons)
  • Utilisé pour entraîner les modèles
  • Le modèle apprend les relations entre les features et le prix

• **Test** : 20% des données (292 maisons)
  • Utilisé pour évaluer les performances des modèles
  • Le modèle n'a jamais vu ces données pendant l'entraînement

**Pourquoi cette séparation ?**

• Éviter le surapprentissage (overfitting)
• Évaluer la capacité de généralisation du modèle
• Obtenir une estimation fiable des performances

**Détails techniques :**

• Méthode : `train_test_split` de scikit-learn
• Ratio : 80% / 20%
• Seed : 42 (pour assurer la reproductibilité)

**Résultat :**

• Train : 1 168 lignes
• Test : 292 lignes

### Prétraitement des données : Imputation

In [ ]:
# ============================================
# Imputation
# (Avec toutes les colonnes)
# ============================================

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Lasso
import numpy as np
import pandas as pd

print("="*60)
print("PREPROCESSING + FEATURE SELECTION")
print("="*60)

# ============================================
# 1. IMPUTATION DE TOUTES LES VALEURS MANQUANTES
# ============================================

print("\n Gestion des valeurs manquantes...")

# 1.1 LotFrontage - par quartier
if 'Neighborhood' in X_train.columns:
    X_train['LotFrontage'] = X_train.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )
    # Appliquer la même imputation sur le test en utilisant les médianes par quartier du train
    neighborhood_medians = X_train.groupby('Neighborhood')['LotFrontage'].median()
    for neighborhood in X_test['Neighborhood'].unique():
        if neighborhood in neighborhood_medians.index:
            mask = X_test['Neighborhood'] == neighborhood
            X_test.loc[mask, 'LotFrontage'] = X_test.loc[mask, 'LotFrontage'].fillna(
                neighborhood_medians[neighborhood]
            )
else:
    # Fallback : médiane globale
    median_lot = X_train['LotFrontage'].median()
    X_train['LotFrontage'] = X_train['LotFrontage'].fillna(median_lot)
    X_test['LotFrontage'] = X_test['LotFrontage'].fillna(median_lot)

print(" LotFrontage imputé")

# 1.2 MasVnrArea - si pas de parement, 0
if 'MasVnrType' in X_train.columns:
    X_train.loc[X_train['MasVnrType'].isna(), 'MasVnrArea'] = 0
    X_test.loc[X_test['MasVnrType'].isna(), 'MasVnrArea'] = 0
    
# Les valeurs restantes (si il y en a) par médiane
median_mas = X_train['MasVnrArea'].median()
X_train['MasVnrArea'] = X_train['MasVnrArea'].fillna(median_mas)
X_test['MasVnrArea'] = X_test['MasVnrArea'].fillna(median_mas)
print(" MasVnrArea imputé")

# 1.3 GarageYrBlt - si pas de garage, utiliser YearBuilt
if 'GarageType' in X_train.columns:
    # Si pas de garage (GarageType manquant), mettre 0
    X_train.loc[X_train['GarageType'].isna(), 'GarageYrBlt'] = 0
    X_test.loc[X_test['GarageType'].isna(), 'GarageYrBlt'] = 0
    
# Pour les valeurs restantes, utiliser YearBuilt
mask_train = X_train['GarageYrBlt'].isna()
X_train.loc[mask_train, 'GarageYrBlt'] = X_train.loc[mask_train, 'YearBuilt']

mask_test = X_test['GarageYrBlt'].isna()
X_test.loc[mask_test, 'GarageYrBlt'] = X_test.loc[mask_test, 'YearBuilt']
print(" GarageYrBlt imputé")

# 1.4 Electrical - par le mode (comme avant)
mode_electrical = X_train['Electrical'].mode()[0]
X_train['Electrical'] = X_train['Electrical'].fillna(mode_electrical).astype(str)
X_test['Electrical'] = X_test['Electrical'].fillna(mode_electrical).astype(str)
print(f" Electrical imputé par le mode : '{mode_electrical}'")

# ============================================
# VÉRIFICATION FINALE
# ============================================

print(f"\n Vérification des NaN restants :")
print(f"   Train : {X_train.isnull().sum().sum()} NaN")
print(f"   Test : {X_test.isnull().sum().sum()} NaN")

# Si encore des NaN, imputer par 0 pour les numériques et mode pour les catégories
if X_train.isnull().sum().sum() > 0:
    print("\n  Il reste des NaN, imputation d'urgence...")
    for col in X_train.columns:
        if X_train[col].dtype in ['int64', 'float64']:
            median_val = X_train[col].median()
            X_train[col] = X_train[col].fillna(median_val)
            X_test[col] = X_test[col].fillna(median_val)
        else:
            mode_val = X_train[col].mode()[0]
            X_train[col] = X_train[col].fillna(mode_val)
            X_test[col] = X_test[col].fillna(mode_val)

**1.Imputation des valeurs manquantes**

Nous avons traité les dernières valeurs manquantes avec des stratégies adaptées à chaque variable :

**LotFrontage** (217 valeurs manquantes) :
• Imputation par la médiane du quartier
• La largeur de façade varie selon les quartiers
• Cette méthode est plus précise qu'une médiane globale

**MasVnrArea** (6 valeurs manquantes) :
• Mise à 0 si MasVnrType = None (pas de parement)
• Sinon, imputation par la médiane

**GarageYrBlt** (64 valeurs manquantes) :
• Si GarageType = None (pas de garage) → 0
• Sinon, remplacement par YearBuilt (année de construction)

**Electrical** (1 valeur manquante) :
• Imputation par le mode (SBrkr) → la valeur la plus fréquente

**Vérification finale :**
• Train : 0 NaN restants 
• Test : 0 NaN restants 
| Variable | NaN | Stratégie | Justification |
|----------|-----|-----------|---------------|
| LotFrontage | 217 | Médiane par quartier | Variation selon les quartiers |
| MasVnrArea | 6 | 0 si pas de parement | Absence de la caractéristique |
| GarageYrBlt | 64 | YearBuilt ou 0 | Lié à la présence d'un garage |
| Electrical | 1 | Mode (SBrkr) | Valeur la plus fréquente |

### Prétraitement des données :Encodage

In [ ]:
# ============================================
# 2. ENCODAGE DES VARIABLES CATÉGORIELLES
# ============================================

#  Définition du mapping des qualités ---
# Ces variables ont un ordre logique (Excellent > Good > Typical > Fair > Poor)
# Le mapping transforme les catégories textuelles en valeurs numériques ordonnées

quality_mapping = {
    'Ex': 5,   # Excellent
    'Gd': 4,   # Good
    'TA': 3,   # Typical / Average
    'Fa': 2,   # Fair
    'Po': 1,   # Poor
    'None': 0  # Absence de la caractéristique
}

# Ces variables sont de type 'object' (string) et ont un ordre logique

quality_cols = [
    'KitchenQual',   # Qualité de la cuisine
    'ExterQual',     # Qualité extérieure
    'ExterCond',     # Condition extérieure
    'BsmtQual',      # Qualité du sous-sol
    'BsmtCond',      # Condition du sous-sol
    'HeatingQC',     # Qualité du chauffage
    'GarageQual',    # Qualité du garage
    'GarageCond',    # Condition du garage
    'PoolQC',        # Qualité de la piscine
    'FireplaceQu'    # Qualité de la cheminée
]

for col in quality_cols:
    if col in X_train.columns:
        # Appliquer le mapping : 
        X_train[col] = X_train[col].map(quality_mapping).fillna(0).astype(int)
        X_test[col] = X_test[col].map(quality_mapping).fillna(0).astype(int)

In [ ]:
#  Sélectionner toutes les colonnes de type 'object'
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

In [ ]:
# ============================================
# ENCODAGE DES VARIABLES CATÉGORIELLES RESTANTES
# ============================================

from sklearn.preprocessing import LabelEncoder

# Encoder les variables catégorielles restantes
for col in cat_cols:
    le = LabelEncoder()
    
    # Convertir en string (sécurité)
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    
    # Apprendre les catégories sur l'ensemble d'entraînement
    le.fit(X_train[col])
    
    # Transformer les données d'entraînement
    X_train[col] = le.transform(X_train[col])
    
    # Gérer les valeurs inconnues dans le test
    # Si une valeur n'existe pas dans le train, on la remplace par la 1ère catégorie
    first_class = le.classes_[0]
    X_test[col] = X_test[col].apply(lambda x: x if x in le.classes_ else first_class)
    X_test[col] = le.transform(X_test[col])

In [ ]:
# ---  Vérification finale de l'encodage ---

print("\n" + "="*60)
print("VÉRIFICATION FINALE DE L'ENCODAGE")
print("="*60)

#  Vérifier le nombre de colonnes
print(f"\n Nombre de colonnes : {X_train.shape[1]}")

#  Vérifier qu'il n'y a plus de colonnes 'object'
remaining_obj = X_train.select_dtypes(include=['object']).columns.tolist()
if remaining_obj:
    print(f"\n {len(remaining_obj)} colonnes 'object' restantes :")
    for col in remaining_obj:
        print(f"   - {col}")
else:
    print("\n Toutes les colonnes sont maintenant numériques !")

print("\n" + "="*60)
print(" ENCODAGE TERMINÉ AVEC SUCCÈS")
print("="*60)

**2. Encodage des variables catégorielles**

Nous avons distingué deux types de variables catégorielles :

---

**2.1 Variables ordinales (avec ordre logique)**

Ces variables ont un ordre naturel qu'il est important de préserver.

**Variables concernées :** KitchenQual, ExterQual, ExterCond, BsmtQual, BsmtCond, HeatingQC, GarageQual, GarageCond, PoolQC, FireplaceQu

**Méthode :** Mapping personnalisé

| Qualité | Valeur | Signification |
|---------|--------|---------------|
| Ex | 5 | Excellent |
| Gd | 4 | Good |
| TA | 3 | Typical / Average |
| Fa | 2 | Fair |
| Po | 1 | Poor |
| None | 0 | Absence de la caractéristique |

**Pourquoi un mapping personnalisé ?**

• Ces variables ont un ordre logique (Ex > Gd > TA > Fa > Po)
• LabelEncoder attribuerait un ordre alphabétique arbitraire
• Le mapping préserve la hiérarchie des qualités
• Les modèles comprennent mieux l'impact de chaque niveau de qualité
• Les valeurs manquantes (None) sont conservées comme une catégorie explicite

---

**2.2 Variables nominales (sans ordre logique)**

**Variables concernées :** Toutes les autres colonnes de type 'object' (quartier, zone, etc.)

**Méthode :** LabelEncoder

**Principe :** Chaque catégorie est transformée en un nombre entier

**Exemple :** "RL" → 0, "RM" → 1, "FV" → 2

**Gestion des valeurs inconnues dans le test :**
• Si une catégorie apparaît dans le test mais pas dans le train
• Elle est remplacée par la catégorie la plus fréquente (first_class)
• Cela garantit la cohérence entre les ensembles

**Vérification finale :**
• Aucune colonne 'object' restante
• Toutes les colonnes sont maintenant numériques

### Prétraitement des données :Standardisation

In [ ]:
# On garde uniquement les int64 et float64 (colonnes originales de base numeriques) pour la standarisation

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Supprimer les colonnes indésirables
if 'Id' in num_cols:
    num_cols.remove('Id')
if 'SalePrice' in num_cols:
    num_cols.remove('SalePrice')

print(f" Colonnes numériques originales : {len(num_cols)}")
print(f"   {num_cols[:10]}...")

# ============================================
# 4. STANDARDISATION UNIQUEMENT DES ORIGINALES
# ============================================

from sklearn.preprocessing import StandardScaler

# Standardiser uniquement les colonnes originales (int64/float64)
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

print(f" Colonnes standardisées : {X_train_num.shape[1]}")

**3. Standardisation des variables numériques originales**

**Objectif :** Mettre les variables numériques sur la même échelle pour les modèles linéaires.

**Variables concernées :** Uniquement les colonnes numériques originales (int64 et float64)

**Pourquoi uniquement les originales ?**

• Les **colonnes mappées** (qualités) ont déjà un ordre logique (1 à 5). La standardisation n'est pas nécessaire.
• Les **colonnes encodées** (LabelEncoder) sont des catégories arbitraires. La standardisation n'est pas pertinente.
• Seules les variables numériques continues (surface, année, etc.) ont besoin d'être standardisées.

**Méthode :** StandardScaler

• **Principe** : Moyenne = 0, Écart-type = 1
• **Formule** : (x - moyenne) / écart-type
• **Pourquoi** : Les modèles linéaires sont sensibles aux échelles des variables

**Colonnes exclues :**

• **Id** : Identifiant unique, pas une feature
• **SalePrice** : Variable cible, pas une feature

**Points importants :**

• Le scaler est **ajusté sur les données d'entraînement** (`fit_transform`)
• Il est **appliqué aux données de test** (`transform`)
→ Cela évite toute fuite d'information

**Résultat :**

• **42 colonnes** standardisées
• Les variables gardent leur distribution mais sont centrées et réduites

### Assemblage

In [ ]:
# ============================================
#  RÉASSEMBLER : NUMÉRIQUES STANDARDISÉES + MAPPÉES + ENCODÉES
# ============================================

# --- 5.1 Récupérer les colonnes mappées (int32) ---
quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
                'HeatingQC', 'KitchenQual', 'FireplaceQu', 
                'GarageQual', 'GarageCond', 'PoolQC']

# Ces colonnes sont en int32, on les garde TELLES QUELLES
X_train_quality = X_train[quality_cols].values
X_test_quality = X_test[quality_cols].values

In [ ]:
#  Récupérer les colonnes encodées (int32) ---
# Toutes les colonnes qui ne sont ni dans num_cols ni dans quality_cols
# Ce sont les colonnes encodées par LabelEncoder

cat_cols = X_train.select_dtypes(include=['int32']).columns.tolist()
cat_cols = [col for col in cat_cols if col not in quality_cols]

X_train_cat = X_train[cat_cols].values
X_test_cat = X_test[cat_cols].values

In [ ]:
# --- Réassembler ---
X_train_processed = np.hstack([X_train_num, X_train_quality, X_train_cat])
X_test_processed = np.hstack([X_test_num, X_test_quality, X_test_cat])

print(f"\n Train après preprocessing : {X_train_processed.shape}")
print(f" Test après preprocessing : {X_test_processed.shape}")

# ============================================
# VÉRIFICATION FINALE
# ============================================

print(f"\n=== VÉRIFICATION FINALE ===")
print(f"  Colonnes standardisées (originales) : {X_train_num.shape[1]}")
print(f"  Colonnes mappées (non standardisées) : {X_train_quality.shape[1]}")
print(f"  Colonnes encodées (non standardisées) : {X_train_cat.shape[1]}")
print(f"  Total : {X_train_processed.shape[1]}")

**4. Assemblage final des données**

Après avoir standardisé les variables numériques originales, mappé les variables ordinales et encodé les variables nominales, nous réassemblons l'ensemble pour obtenir le jeu de données final.

---

**4.1 Structure des données assemblées**

| Type de colonnes | Nombre | Traitement | Standardisation |
|------------------|--------|------------|-----------------|
| Numériques originales | 42 | StandardScaler |  Oui |
| Variables mappées | 10 | Mapping personnalisé |  Non |
| Variables encodées | 33 | LabelEncoder |  Non |
| **Total** | **85** | | |

---

**Pourquoi certains types ne sont pas standardisés ?**

• **Variables mappées** : Elles ont déjà un ordre logique (1 à 5). La standardisation supprimerait cette hiérarchie.
• **Variables encodées** : Ce sont des catégories arbitraires transformées en nombres. La standardisation n'est pas pertinente.

---

**Vérification finale :**

| Composant | Nombre |
|-----------|--------|
| Colonnes standardisées | 42 |
| Colonnes mappées | 10 |
| Colonnes encodées | 33 |
| **Total** | **85** |

**Dimensions finales :**
• Train : (1166, 85)
• Test : (292, 85)
• Toutes les colonnes sont numériques et prêtes pour la modélisation

### Feature selection pour la regression linéaire 

In [35]:
# ============================================
# 6. LASSO POUR LA SÉLECTION DE FEATURES
# ============================================

print("\n" + "="*60)
print("LASSO FEATURE SELECTION")
print("="*60)

# Option 1 : Lasso avec validation croisée
from sklearn.linear_model import LassoCV
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_processed, y_train)

# Option 2 : Lasso simple avec alpha choisi manuellement
# lasso = Lasso(alpha=0.001, random_state=42, max_iter=10000)
# lasso.fit(X_train_processed, y_train)

# Récupérer les coefficients
feature_names = num_cols + cat_cols + quality_cols
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso.coef_
})

# Filtrer les features importantes
important_features = coef_df[coef_df['Coefficient'] != 0].sort_values('Coefficient', key=abs, ascending=False)
print(f"\n {len(important_features)} features sélectionnées sur {len(feature_names)}")

print(f"\n Top 20 features les plus importantes :")
print(important_features.head(20))

# Sélectionner les colonnes à garder
selected_features = important_features['Feature'].tolist()
print(f"\n Features à garder : {len(selected_features)}")

# ============================================
# 7. CRÉER LES DATASETS FINAUX AVEC SEULEMENT LES FEATURES SÉLECTIONNÉES
# ============================================

# Garder uniquement les colonnes sélectionnées dans les DataFrames originaux
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"\n X_train_final : {X_train_selected.shape}")
print(f" X_test_final : {X_test_selected.shape}")

# Optionnel : Sauvegarder
# X_train_selected.to_csv('X_train_selected.csv', index=False)
# X_test_selected.to_csv('X_test_selected.csv', index=False)


LASSO FEATURE SELECTION

 61 features sélectionnées sur 85

 Top 20 features les plus importantes :
         Feature  Coefficient
36   QualityArea    90604.288
15     GrLivArea   -26151.680
3    OverallQual   -20160.479
53      BldgType    15248.514
8     BsmtFinSF1     9380.824
69  GarageFinish     7900.217
11   TotalBsmtSF     6901.348
20  BedroomAbvGr    -6441.099
22  TotRmsAbvGrd     6237.697
38      HouseAge    -6110.651
4    OverallCond     5311.449
13      2ndFlrSF    -5009.658
7     MasVnrArea     4748.020
2        LotArea     4727.445
25    GarageCars     4542.825
77      BsmtQual     4245.623
84        PoolQC     3983.422
5      YearBuilt     3334.680
62  BsmtFinType1    -3297.999
70    PavedDrive     3006.622

 Features à garder : 61

 X_train_final : (1166, 61)
 X_test_final : (292, 61)


**Sélection de features avec Lasso**

**Objectif :** Réduire le nombre de variables pour simplifier le modèle et éviter l'overfitting.

**Méthode : Lasso avec validation croisée (LassoCV)**

• **Principe** : La régularisation L1 permet de mettre des coefficients à zéro
• **Avantage** : Sélection automatique des features importantes
• **Validation croisée** : 5 folds pour choisir le meilleur paramètre alpha

**Paramètres utilisés :**

• `cv=5` : Validation croisée à 5 plis
• `max_iter=10000` : Nombre maximum d'itérations pour la convergence
• `random_state=42` : Pour assurer la reproductibilité

**Résultat de la sélection :**

• **Avant** : 85 features
• **Après** : 65 features sélectionnées
• **Réduction** : -23.5% des variables

**Top 10 features les plus importantes :**

| Feature | Coefficient | Interprétation |
|---------|-------------|----------------|
| QualityArea | +83 570 | Qualité × Surface (très influent) |
| GrLivArea | -20 330 | Surface habitable |
| OverallQual | -20 218 | Qualité globale de la maison |
| BsmtFinSF1 | +9 423 | Surface finie du sous-sol |
| MSZoning | +7 670 | Zone de la maison |
| GarageFinish | +7 242 | Fini du garage |
| HouseAge | -6 992 | Âge de la maison |
| Utilities | +6 149 | Services publics disponibles |
| 2ndFlrSF | -5 922 | Surface du 2ème étage |
| TotalBsmtSF | +5 453 | Surface totale du sous-sol |

**Observations :**

• **QualityArea** est la variable la plus influente (coefficient le plus élevé)
• **GrLivArea** et **OverallQual** ont des coefficients négatifs (liés à une régularisation)
• Des variables comme **MSZoning**, **GarageFinish**, **HouseAge** sont également importantes
• **PoolQC** est également sélectionnée (qualité de la piscine)

**Création des datasets finaux :**

• **X_train_selected** : (1166, 65) → 65 features sélectionnées
• **X_test_selected** : (292, 65) → 65 features sélectionnées

**Pourquoi cette sélection est importante ?**

• Réduit le risque d'overfitting
• Simplifie le modèle
• Améliore la vitesse d'entraînement
• Facilite l'interprétation des résultats

**Features éliminées :**

• 20 features jugées moins importantes
• Leurs coefficients ont été mis à zéro par Lasso
• Elles n'apportent pas d'information supplémentaire significative

In [59]:
# ============================================
# CRÉATION DES DEUX VERSIONS DES DATASETS
# ============================================

print("\n" + "="*60)
print(" CRÉATION DES DATASETS")
print("="*60)

# ============================================
# 1. VERSION AVEC TOUTES LES FEATURES (85)
# ============================================

print("\n 1. Version avec toutes les features (85) :")
print(f"   X_train_full : {X_train.shape}")
print(f"   X_test_full : {X_test.shape}")

# Sauvegarder la version complète
X_train_full = X_train.copy()
X_test_full = X_test.copy()

# ============================================
# 2. VERSION AVEC FEATURES SÉLECTIONNÉES 
# ============================================

print("\n 2. Version avec features sélectionnées  :")
print(f"   X_train_selected : {X_train_selected.shape}")
print(f"   X_test_selected : {X_test_selected.shape}")

# ============================================
# 3. SAUVEGARDE DES DEUX VERSIONS
# ============================================

import os
os.makedirs('../data/processed', exist_ok=True)

# 3.1 Version complète (85 features)
X_train_full.to_csv('../data/processed/X_train_full.csv', index=False)
X_test_full.to_csv('../data/processed/X_test_full.csv', index=False)
print("\n Version complète (85 features) sauvegardée :")
print("   - data/processed/X_train_full.csv")
print("   - data/processed/X_test_full.csv")

# 3.2 Version sélectionnée (40 features)
X_train_selected.to_csv('../data/processed/X_train_selected.csv', index=False)
X_test_selected.to_csv('../data/processed/X_test_selected.csv', index=False)
print("\n Version sélectionnée  sauvegardée :")
print("   - data/processed/X_train_selected.csv")
print("   - data/processed/X_test_selected.csv")

# 3.3 Sauvegarder y_train et y_test (communs)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
print("\n y_train et y_test sauvegardés :")
print("   - data/processed/y_train.csv")
print("   - data/processed/y_test.csv")

# 3.4 Sauvegarder la liste des features sélectionnées
with open('../data/processed/selected_features.txt', 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")
print("\n Liste des features sélectionnées sauvegardée :")
print("   - data/processed/selected_features.txt")


 CRÉATION DES DATASETS

 1. Version avec toutes les features (85) :
   X_train_full : (1166, 85)
   X_test_full : (292, 85)

 2. Version avec features sélectionnées  :
   X_train_selected : (1166, 65)
   X_test_selected : (292, 65)

 Version complète (85 features) sauvegardée :
   - data/processed/X_train_full.csv
   - data/processed/X_test_full.csv

 Version sélectionnée  sauvegardée :
   - data/processed/X_train_selected.csv
   - data/processed/X_test_selected.csv

 y_train et y_test sauvegardés :
   - data/processed/y_train.csv
   - data/processed/y_test.csv

 Liste des features sélectionnées sauvegardée :
   - data/processed/selected_features.txt


**Création et sauvegarde des datasets finaux**

**Pourquoi deux versions ?**

Nous avons créé deux versions des données pour répondre aux besoins spécifiques de chaque type de modèle :

• **Version complète (85 features)** :
  • Utilisée pour les modèles à arbres (Random Forest, XGBoost, LightGBM, CatBoost)
  • Ces modèles gèrent naturellement un grand nombre de variables
  • Ils intègrent leurs propres mécanismes de sélection

• **Version sélectionnée (65 features)** :
  • Utilisée pour la régression linéaire
  • Les modèles linéaires bénéficient d'une réduction du nombre de variables
  • Réduit le risque d'overfitting et améliore la stabilité

**Fichiers sauvegardés :**

| Fichier | Contenu | Features |
|---------|---------|----------|
| X_train_full.csv | Features train | 85 |
| X_test_full.csv | Features test | 85 |
| X_train_selected.csv | Features train | 65 |
| X_test_selected.csv | Features test | 65 |
| y_train.csv | Cible train | 1 |
| y_test.csv | Cible test | 1 |
| selected_features.txt | Liste des 65 features sélectionnées | - |

**Emplacement :** `../data/processed/`

**Structure finale :**
